To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm==0.8.2
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm==0.8.2


In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

### Unsloth

Load up `Llama 3.1 8B Instruct`, and set parameters

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.6, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

==((====))==  Unsloth 2025.5.9: Fast Qwen2 patching. Transformers: 4.52.2. vLLM: 0.8.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit with actual GPU utilization = 42.05%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 4096. Num Sequences = 192.
Unsloth: vLLM's KV Cache can use up to 4.93 GB. Also swap space = 0 GB.
WARNING 06-01 16:23:32 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 06-01 16:23:32 [config.py:585] This model supports multiple tasks: {'score', 'generate', 'reward', 'embe

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 06-01 16:23:53 [model_runner.py:1146] Model loading took 1.4944 GB and 16.873452 seconds
INFO 06-01 16:23:55 [worker.py:267] Memory profiling takes 1.62 seconds
INFO 06-01 16:23:55 [worker.py:267] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.42) = 6.20GiB
INFO 06-01 16:23:55 [worker.py:267] model weights take 1.49GiB; non_torch_memory takes 0.00GiB; PyTorch activation peak memory takes 1.05GiB; the rest of the memory reserved for KV Cache is 3.66GiB.
INFO 06-01 16:23:56 [executor_base.py:111] # cuda blocks: 8561, # CPU blocks: 0
INFO 06-01 16:23:56 [executor_base.py:116] Maximum concurrency for 4096 tokens per request: 33.44x
INFO 06-01 16:23:56 [model_runner.py:1442] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing

Capturing CUDA graph shapes: 100%|██████████| 27/27 [00:39<00:00,  1.48s/it]

INFO 06-01 16:24:36 [model_runner.py:1570] Graph capturing finished in 40 secs, took 0.06 GiB
INFO 06-01 16:24:36 [llm_engine.py:447] init engine (profile, create kv cache, warmup model) took 42.79 seconds


Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'post_feedforward_layernorm', 'q_norm', 'k_norm']
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'post_feedforward_layernorm', 'q_norm', 'k_norm']


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [ ]:
import re
from datasets import load_dataset, Dataset
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from bs4 import BeautifulSoup

# Load and prep dataset
# SYSTEM_PROMPT = """
# Respond in the following format:
# <reasoning>
# ...
# </reasoning>
# <answer>
# ...
# </answer>
# """
SYSTEM_PROMPT = """
You are an expert frontend developer. Generate valid, integrated, semantic HTML/CSS/JS for the requested web page or component.
Use proper HTML5 structure with appropriate semantic elements.
Please consider following points:
- DOCTYPE declaration
- html tag with lang attribute
- head section with viewport meta tag
- semantic structure (header, main, footer, etc.)
- proper heading hierarchy (h1, h2, etc.)
- CSS for styling
- JS for interactivity
- Integrate HTML, CSS, JS in one HTML

You have to reason yourself and give the process of reasoning in <reasoning> and give the html code in <answer>
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>

"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

import pandas as pd

import pandas as pd
from datasets import Dataset
import json
def get_ui_data(split="train") -> Dataset:
    with open("/content/ui_design_results.json", "r", encoding="utf-8") as f:
      json_data = json.load(f)
    #df = pd.read_csv(f"UI_REASONING_{split}.csv")
    df = pd.DataFrame(json_data)

    # Rename columns to match expected format
    df = df.rename(columns={"query": "question", "html": "answer"})
    # Apply transformation to each row
    df['prompt'] = df.apply(lambda x: [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': x['question']}
    ], axis=1)

    # df['answer'] = df['answer'].apply(extract_hash_answer)

    # Keep only 'prompt' and 'answer' columns
    return Dataset.from_pandas(df[['prompt', 'answer']])

print(get_ui_data)
# uncomment middle messages for 1-shot prompting
# def get_gsm8k_questions(split = "train") -> Dataset:
#     data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
#     data = data.map(lambda x: { # type: ignore
#         'prompt': [
#             {'role': 'system', 'content': SYSTEM_PROMPT},
#             {'role': 'user', 'content': x['question']}
#         ],
#         'answer': extract_hash_answer(x['answer'])
#     }) # type: ignore
#     return data # type: ignore

# dataset = get_gsm8k_questions()
dataset = get_ui_data()

# # Reward functions
# def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
#     responses = [completion[0]['content'] for completion in completions]
#     q = prompts[0][-1]['content']
#     extracted_responses = [extract_xml_answer(r) for r in responses]
#     print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
#     return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

# def int_reward_func(completions, **kwargs) -> list[float]:
#     responses = [completion[0]['content'] for completion in completions]
#     extracted_responses = [extract_xml_answer(r) for r in responses]
#     return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

# def strict_format_reward_func(completions, **kwargs) -> list[float]:
#     """Reward function that checks if the completion has a specific format."""
#     pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
#     responses = [completion[0]["content"] for completion in completions]
#     matches = [re.match(pattern, r) for r in responses]
#     return [0.5 if match else 0.0 for match in matches]

# def soft_format_reward_func(completions, **kwargs) -> list[float]:
#     """Reward function that checks if the completion has a specific format."""
#     pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
#     responses = [completion[0]["content"] for completion in completions]
#     matches = [re.match(pattern, r) for r in responses]
#     return [0.5 if match else 0.0 for match in matches]


# HTML Structure Reward Function
# def html_structure_reward_func(completions, **kwargs) -> list[float]:
#     """Reward function that evaluates HTML structure quality using HTMLAnalyzer"""
#     responses = [completion[0]["content"] for completion in completions]
#     scores = []

#     for html_content in responses:
#         # Create a simple HTMLAnalyzer instance to evaluate the HTML
#         try:
#             # Use html.parser for better compatibility
#             soup = BeautifulSoup(html_content, 'html.parser')
#             score = 100.0

#             # Check for doctype
#             if not html_content.lower().startswith('<!doctype'):
#                 score -= 10.0

#             # Check for lang attribute
#             html_tag = soup.find('html')
#             if not html_tag or not html_tag.get('lang'):
#                 score -= 5.0

#             # Check for viewport meta tag
#             viewport_meta = soup.find('meta', attrs={'name': 'viewport'})
#             if not viewport_meta:
#                 score -= 5.0

#             # Check for proper heading structure
#             headings = soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])
#             if not headings:
#                 score -= 5.0
#             else:
#                 # Check for h1
#                 if not soup.find('h1'):
#                     score -= 5.0

#             # Check for semantic elements
#             semantic_tags = ['header', 'nav', 'main', 'section', 'article', 'aside', 'footer']
#             semantic_elements = soup.find_all(semantic_tags)
#             if len(semantic_elements) < 2:
#                 score -= 5.0

#             # Check for divitis (excessive div usage)
#             div_count = len(soup.find_all('div'))
#             total_elements = len(list(soup.find_all()))
#             if div_count > 0 and total_elements > 0:
#                 div_ratio = div_count / total_elements
#                 if div_ratio > 0.7:  # If more than 70% of elements are divs
#                     score -= 10.0

#             # Check for proper form elements
#             forms = soup.find_all('form')
#             if forms:
#                 for form in forms:
#                     inputs = form.find_all('input')
#                     if inputs:
#                         for input_tag in inputs:
#                             if not input_tag.get('id'):
#                                 score -= 2.0

#             # Normalize score to [0, 1] range for GRPO
#             normalized_score = max(0.0, min(score, 100.0)) / 100.0
#             scores.append(normalized_score)

#         except Exception as e:
#             # If parsing fails, give a low score
#             print(f"Error analyzing HTML: {str(e)}")
#             scores.append(0.0)

#     # Print a sample for debugging
#     if scores:
#         print('-'*20, f"HTML Score Sample:\nContent:\n{responses[0][:200]}...\nScore: {scores[0]}")

#     return scores

# # Function to check if response contains valid HTML
# def valid_html_reward_func(completions, **kwargs) -> list[float]:
#     responses = [completion[0]["content"] for completion in completions]
#     scores = []

#     for html_content in responses:
#         try:
#             soup = BeautifulSoup(html_content, 'html.parser')
#             # Check if there's at least a basic HTML structure
#             if soup.find('html') and soup.find('body'):
#                 scores.append(0.5)
#             else:
#                 scores.append(0.0)
#         except Exception:
#             scores.append(0.0)

#     return scores

# # Function to check semantic richness (variety of semantic elements)
# def semantic_richness_reward_func(completions, **kwargs) -> list[float]:
#     responses = [completion[0]["content"] for completion in completions]
#     scores = []

#     semantic_tags = ['header', 'nav', 'main', 'section', 'article', 'aside', 'footer',
#                      'figure', 'figcaption', 'details', 'summary', 'mark', 'time']

#     for html_content in responses:
#         try:
#             soup = BeautifulSoup(html_content, 'html.parser')
#             # Count unique semantic elements
#             found_semantic_tags = set()
#             for tag in semantic_tags:
#                 if soup.find(tag):
#                     found_semantic_tags.add(tag)

#             # Calculate score based on variety (0.5 is max score)
#             semantic_score = min(0.5, len(found_semantic_tags) * 0.1)
#             scores.append(semantic_score)
#         except Exception:
#             scores.append(0.0)

#     return scores


def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

def color_score(completions, **kwargs):
    completions = completions
    kwargs = kwargs
    responses = [completion[0]["content"] for completion in completions]
    from bs4 import BeautifulSoup
    import re
    import colorsys
    import numpy as np
    from collections import Counter
    import math

    def extract_colors(css_text):
        hex_colors = re.findall(r'#(?:[0-9a-fA-F]{3}){1,2}', css_text)
        rgb_colors = re.findall(r'rgb[a]?\(([^)]+)\)', css_text)
        rgb_colors = ['rgb(' + c + ')' for c in rgb_colors]
        return hex_colors + rgb_colors

    def color_to_hsl(color):
        try:
            if color.startswith('#'):
                hex_color = color.lstrip('#')
                if len(hex_color) == 3:
                    hex_color = ''.join([c*2 for c in hex_color])
                r, g, b = [int(hex_color[i:i+2], 16)/255.0 for i in (0, 2, 4)]
            elif color.startswith('rgb'):
                nums = [int(n.strip()) for n in re.findall(r'\d+', color)[:3]]
                r, g, b = [n / 255.0 for n in nums]
            else:
                return None
            return colorsys.rgb_to_hls(r, g, b)
        except:
            return None

    def compute_color_harmony_score(colors):
        hsl_colors = [color_to_hsl(c) for c in colors]
        hsl_colors = [c for c in hsl_colors if c is not None]
        if len(hsl_colors) < 2:
            return 50
        hues = [c[0] for c in hsl_colors]
        lightness = [c[1] for c in hsl_colors]
        saturation = [c[2] for c in hsl_colors]
        hue_var = np.var(hues)
        light_var = np.var(lightness)
        sat_var = np.var(saturation)
        hue_score = 100 - min(hue_var * 200, 100)
        light_score = 100 - min(light_var * 300, 100)
        sat_score = 100 - min(sat_var * 300, 100)
        return round((hue_score * 0.4 + light_score * 0.3 + sat_score * 0.3), 2)

    def evaluate_design_from_html(html_text):
        soup = BeautifulSoup(html_text, 'html.parser')
        css_text = ''.join([tag.string or '' for tag in soup.find_all('style')])
        inline_styles = ' '.join([tag.get('style', '') for tag in soup.find_all(style=True)])
        combined_css = css_text + ' ' + inline_styles

        colors = extract_colors(combined_css)
        color_score = compute_color_harmony_score(colors)
        layout_score = compute_layout_score(soup)
        responsive_score = evaluate_responsiveness_score(soup, html_text)

        # 새로운 가중치 기반 총점 (각 3분할)
        total_score = round((color_score * 0.35 + layout_score * 0.35 + responsive_score * 0.30), 2)

        return color_score
        return {
            "color_harmony": color_score,
            "layout": layout_score,
            "responsiveness": responsive_score,
            "total_score": total_score
        }

    results = []
    for html_content in responses:
        results.append(evaluate_design_from_html(html_content))
    return results


def layout_score(completions, **kwargs):
    completions = completions
    kwargs = kwargs
    responses = [completion[0]["content"] for completion in completions]
    from bs4 import BeautifulSoup
    import re
    import colorsys
    import numpy as np
    from collections import Counter
    import math

    def compute_layout_score(soup):
        divs = soup.find_all(['div', 'section', 'article', 'main', 'aside'])
        if not divs:
            return 50
        depths, widths, heights = [], [], []
        for div in divs:
            depth = len(list(div.parents))
            style = div.get('style', '')
            width = re.search(r'width\s*:\s*(\d+)', style)
            height = re.search(r'height\s*:\s*(\d+)', style)
            depths.append(depth)
            if width:
                widths.append(int(width.group(1)))
            if height:
                heights.append(int(height.group(1)))
        depth_score = 100 - min(np.std(depths) * 10, 50)
        width_score = 100 - min(np.std(widths) if widths else 50, 50)
        height_score = 100 - min(np.std(heights) if heights else 50, 50)
        return round((depth_score * 0.4 + width_score * 0.3 + height_score * 0.3), 2)

    def evaluate_design_from_html(html_text):
        soup = BeautifulSoup(html_text, 'html.parser')
        css_text = ''.join([tag.string or '' for tag in soup.find_all('style')])
        inline_styles = ' '.join([tag.get('style', '') for tag in soup.find_all(style=True)])
        combined_css = css_text + ' ' + inline_styles

        colors = extract_colors(combined_css)
        color_score = compute_color_harmony_score(colors)
        layout_score = compute_layout_score(soup)
        responsive_score = evaluate_responsiveness_score(soup, html_text)

        # 새로운 가중치 기반 총점 (각 3분할)
        total_score = round((color_score * 0.35 + layout_score * 0.35 + responsive_score * 0.30), 2)

        return layout_score
        return {
            "color_harmony": color_score,
            "layout": layout_score,
            "responsiveness": responsive_score,
            "total_score": total_score
        }

    results = []
    for html_content in responses:
        results.append(evaluate_design_from_html(html_content))
    return results

def responsive_score(completions, **kwargs):
    completions = completions
    kwargs = kwargs
    responses = [completion[0]["content"] for completion in completions]
    from bs4 import BeautifulSoup
    import re
    import colorsys
    import numpy as np
    from collections import Counter
    import math
    def evaluate_responsiveness_score(soup, html_text):
        css_text = ''.join([tag.string or '' for tag in soup.find_all('style')])
        inline_styles = ' '.join([tag.get('style', '') for tag in soup.find_all(style=True)])
        full_style = (css_text + ' ' + inline_styles + ' ' + html_text).lower()

        # 1. 구조 기반 점수 (최대 50점)
        structural_score = 0
        if soup.find('meta', attrs={"name": "viewport"}):
            structural_score += 25
        if '@media' in full_style:
            structural_score += 25

        # 2. 고정 단위 감점 (최대 감점 20점)
        fixed_units = re.findall(r'(?:style\s*=\s*["\'][^"\']*?)(\d+(px|pt|cm|mm))', full_style)
        fixed_penalty = len(fixed_units) * 1.5
        fixed_penalty += full_style.count('px') * 0.5
        fixed_penalty = min(fixed_penalty, 20)

        # 3. 반응형 단위 보너스 (최대 30점)
        responsive_bonus = 0
        for unit in ['%', 'vw', 'vh', 'em', 'rem']:
            responsive_bonus += full_style.count(unit) * 0.2
        responsive_bonus = min(responsive_bonus, 30)

        # 종합 점수
        score = structural_score - fixed_penalty + responsive_bonus
        return round(max(0, min(score, 100)), 2)

    def evaluate_design_from_html(html_text):
        soup = BeautifulSoup(html_text, 'html.parser')
        css_text = ''.join([tag.string or '' for tag in soup.find_all('style')])
        inline_styles = ' '.join([tag.get('style', '') for tag in soup.find_all(style=True)])
        combined_css = css_text + ' ' + inline_styles

        colors = extract_colors(combined_css)
        color_score = compute_color_harmony_score(colors)
        layout_score = compute_layout_score(soup)
        responsive_score = evaluate_responsiveness_score(soup, html_text)

        # 새로운 가중치 기반 총점 (각 3분할)
        total_score = round((color_score * 0.35 + layout_score * 0.35 + responsive_score * 0.30), 2)

        return layout_score
        return {
            "color_harmony": color_score,
            "layout": layout_score,
            "responsiveness": responsive_score,
            "total_score": total_score
        }

    results = []
    for html_content in responses:
        results.append(evaluate_design_from_html(html_content))
    return results

<function get_ui_data at 0x7fcdf033c900>


In [ ]:
import pprint
print(dataset)
pprint.pprint(dataset['prompt'][0])
pprint.pprint(dataset['answer'][0])
# pprint.pprint(dataset['prompt'][5])

Dataset({
    features: ['prompt', 'answer'],
    num_rows: 772
})
[{'content': '\n'
             'You are an expert frontend developer. Generate valid, '
             'integrated, semantic HTML/CSS/JS for the requested web page or '
             'component.\n'
             'Use proper HTML5 structure with appropriate semantic elements.\n'
             'Please consider following points:\n'
             '- DOCTYPE declaration\n'
             '- html tag with lang attribute\n'
             '- head section with viewport meta tag\n'
             '- semantic structure (header, main, footer, etc.)\n'
             '- proper heading hierarchy (h1, h2, etc.)\n'
             '- CSS for styling\n'
             '- JS for interactivity\n'
             '- Integrate HTML, CSS, JS in one HTML\n'
             '\n'
             'You have to reason yourself and give the process of reasoning in '
             '<reasoning> and give the html code in <answer>\n'
             'Respond in the following format:

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
max_prompt_length = 256

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 6, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 250,
    save_steps = 250,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 6


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        color_score,
        layout_score,
        responsive_score,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

TypeError: _UnslothGRPOTrainer._get_train_sampler() takes 1 positional argument but 2 were given

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>


In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from bs4 import BeautifulSoup

# System prompt for HTML generation
SYSTEM_PROMPT = """
You are an expert HTML developer. Generate valid, semantic HTML for the requested web page or component.
Use proper HTML5 structure with appropriate semantic elements.
Always include:
- DOCTYPE declaration
- html tag with lang attribute
- head section with viewport meta tag
- semantic structure (header, main, footer, etc.)
- proper heading hierarchy (h1, h2, etc.)
"""

# Define HTML dataset preparation
def get_ui_data(split="train") -> Dataset:
    df = pd.read_csv(f"UI_REASONING_{split}.csv")

    # Apply transformation to each row
    df['prompt'] = df.apply(lambda x: [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': x['question']}
    ], axis=1)

    # df['answer'] = df['answer'].apply(extract_hash_answer)

    # Keep only 'prompt' and 'answer' columns
    return Dataset.from_pandas(df[['prompt', 'answer']])


# HTML Structure Reward Function
def html_structure_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that evaluates HTML structure quality using HTMLAnalyzer"""
    responses = [completion[0]["content"] for completion in completions]
    scores = []

    for html_content in responses:
        # Create a simple HTMLAnalyzer instance to evaluate the HTML
        try:
            # Use html.parser for better compatibility
            soup = BeautifulSoup(html_content, 'html.parser')
            score = 100.0

            # Check for doctype
            if not html_content.lower().startswith('<!doctype'):
                score -= 10.0

            # Check for lang attribute
            html_tag = soup.find('html')
            if not html_tag or not html_tag.get('lang'):
                score -= 5.0

            # Check for viewport meta tag
            viewport_meta = soup.find('meta', attrs={'name': 'viewport'})
            if not viewport_meta:
                score -= 5.0

            # Check for proper heading structure
            headings = soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])
            if not headings:
                score -= 5.0
            else:
                # Check for h1
                if not soup.find('h1'):
                    score -= 5.0

            # Check for semantic elements
            semantic_tags = ['header', 'nav', 'main', 'section', 'article', 'aside', 'footer']
            semantic_elements = soup.find_all(semantic_tags)
            if len(semantic_elements) < 2:
                score -= 5.0

            # Check for divitis (excessive div usage)
            div_count = len(soup.find_all('div'))
            total_elements = len(list(soup.find_all()))
            if div_count > 0 and total_elements > 0:
                div_ratio = div_count / total_elements
                if div_ratio > 0.7:  # If more than 70% of elements are divs
                    score -= 10.0

            # Check for proper form elements
            forms = soup.find_all('form')
            if forms:
                for form in forms:
                    inputs = form.find_all('input')
                    if inputs:
                        for input_tag in inputs:
                            if not input_tag.get('id'):
                                score -= 2.0

            # Normalize score to [0, 1] range for GRPO
            normalized_score = max(0.0, min(score, 100.0)) / 100.0
            scores.append(normalized_score)

        except Exception as e:
            # If parsing fails, give a low score
            print(f"Error analyzing HTML: {str(e)}")
            scores.append(0.0)

    # Print a sample for debugging
    if scores:
        print('-'*20, f"HTML Score Sample:\nContent:\n{responses[0][:200]}...\nScore: {scores[0]}")

    return scores

# Function to check if response contains valid HTML
def valid_html_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]["content"] for completion in completions]
    scores = []

    for html_content in responses:
        try:
            soup = BeautifulSoup(html_content, 'html.parser')
            # Check if there's at least a basic HTML structure
            if soup.find('html') and soup.find('body'):
                scores.append(0.5)
            else:
                scores.append(0.0)
        except Exception:
            scores.append(0.0)

    return scores

# Function to check semantic richness (variety of semantic elements)
def semantic_richness_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]["content"] for completion in completions]
    scores = []

    semantic_tags = ['header', 'nav', 'main', 'section', 'article', 'aside', 'footer',
                     'figure', 'figcaption', 'details', 'summary', 'mark', 'time']

    for html_content in responses:
        try:
            soup = BeautifulSoup(html_content, 'html.parser')
            # Count unique semantic elements
            found_semantic_tags = set()
            for tag in semantic_tags:
                if soup.find(tag):
                    found_semantic_tags.add(tag)

            # Calculate score based on variety (0.5 is max score)
            semantic_score = min(0.5, len(found_semantic_tags) * 0.1)
            scores.append(semantic_score)
        except Exception:
            scores.append(0.0)

    return scores

# Load dataset
dataset = get_ui_data()

# Model configuration
model_name = "Qwen/Qwen2.5-7B-Instruct"  # Or another suitable model
output_dir = "outputs/HTML-GRPO"
run_name = "HTML-Structure-GRPO"

# Configure GRPO training
training_args = GRPOConfig(
    output_dir=output_dir,
    run_name=run_name,
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    logging_steps=1,
    bf16=torch.cuda.is_available(),  # Use bf16 if available
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=16,
    max_prompt_length=256,
    max_completion_length=2048,  # HTML can be lengthy
    num_train_epochs=1,
    save_steps=100,
    max_grad_norm=0.1,
    report_to="wandb",
    log_on_each_node=False,
)

# Configure LoRA fine-tuning
peft_config = LoraConfig(
    r=16,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    task_type="CAUSAL_LM",
    lora_dropout=0.05,
)

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    attn_implementation="flash_attention_2" if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Create GRPO trainer with HTML-specific reward functions
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        html_structure_reward_func,  # Main HTML structure score
        valid_html_reward_func,      # Valid HTML check
        semantic_richness_reward_func  # Semantic richness check
    ],
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config
)

# Start training
trainer.train()

In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the original pretrained model
model_name = "Tesslate/UIGEN-T2-7B"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto").eval()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Example prompt (same as fine-tuned test)
prompt = "Create a responsive HTML/CSS/JS page for coffee ordering web page."

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2056,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95
    )

# Decode and print output
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=== ORIGINAL MODEL OUTPUT ===")
print(decoded_output)

# Save output to file
with open("output.txt", "w") as f:
    f.write(decoded_output)
